# WeldFatigueChecker · 自采 40 张增量训练（AI Studio Notebook）
把 `raw_mine` 自采焊缝图（40 张）并入基于 120 轮权重的检测模型，并**采用持续学习模式，历史数据绝不重跑**。

## 前置准备（在 AI Studio 网页操作）
1. 把本仓库的 `ml/` 目录整库上传到 AI Studio 工作区 `/home/aistudio/work/`（即 `work/ml/weld_train.py` + `work/ml/raw_mine/`）。
   - 若只上传了 `raw_mine`，其他源(源1~4)会自动跳过，不影响——本次只在 40 张自采图上基于 120 权重微调。
2. 把 120 轮权重 `.pt` 上传到 `work/`，例如：
   - 之前按备份提示下载过的 `weld_defect_5cls_last_backup.pt`，或
   - AI Studio `work/runs/detect/weld_defect_5cls*/weights/last.pt`（若 work/ 未被清）。
3. 新建 Notebook，上传并**从上到下依次运行**本文件的所有 cell。

In [ ]:
!pip install -q ultralytics pyyaml
import torch, ultralytics
print('torch', torch.__version__, '| ultralytics', ultralytics.__version__)

In [ ]:
ML_DIR = '/home/aistudio/work/ml'                 # ← 按实际上传位置修改（weld_train.py 所在目录）
CKPT   = '/home/aistudio/work/weld_defect_5cls_last_backup.pt'  # ← 120 权重绝对路径；若在 runs/ 下写 'runs/detect/weld_defect_5cls/weights/last.pt'
EPOCHS = 40   # 首次只训 40 张自采图：30~40 轮足够；若你同时上传了源1~4全量数据，改 120
%cd {ML_DIR}

In [ ]:
import os
print('weld_train.py 存在 :', os.path.isfile('weld_train.py'))
print('raw_mine 图片数   :', len([f for f in os.listdir('raw_mine/images') if f.lower().endswith(('.jpg','.png','.jpeg'))]))
print('raw_mine 标签数   :', len([f for f in os.listdir('raw_mine/labels') if f.lower().endswith('.txt')]))
print('120 权重存在      :', os.path.isfile(CKPT))

In [ ]:
# 持续学习：首次（无 manifest）把 40 张当新增全量训一次建立基线；之后重跑只训新增，历史不重跑
!python weld_train.py --incremental --init-from {CKPT} --epochs {EPOCHS}

In [ ]:
import glob, os
runs = sorted(glob.glob('runs/detect/weld_defect_5cls*/weights/best.pt'))
print('最新训练产物      :', runs[-1] if runs else '未找到')
print('mlpackage 已导出  :', os.path.isdir('WeldDefectModel.mlpackage'))
print('已训图清单 manifest:', os.path.isfile('dataset/.trained_manifest.txt'))
if os.path.isfile('dataset/.trained_manifest.txt'):
    with open('dataset/.trained_manifest.txt') as f:
        print('  已训图数        :', len([l for l in f if l.strip()]))

In [ ]:
from IPython.display import FileLink
display(FileLink('WeldDefectModel.mlpackage'))
print('建议同时把 work/weld_defect_5cls_last_backup.pt 下载到本机兜底（每个 epoch 已自动备份）')

## 以后继续补充自采图（持续学习，历史绝不重跑）
1. 把新图丢进 `raw_mine/images/` + 同名标签 `raw_mine/labels/`（YOLO 5 类格式）。
2. 重跑上面的**训练 Cell**，但去掉 `--init-from`，改为：
   ```python
   !python weld_train.py --incremental
   ```
   - 脚本自动借上次 `last.pt` 权重，只训【新增图】+ 抽 ≤200 张历史图回放防遗忘，默认 30 轮，数十分钟级。
3. 下载新 `WeldDefectModel.mlpackage` 回 Xcode 工程替换即可。

⚠️ 不要把 `dataset/` 整个删除（会丢 `.trained_manifest.txt`）；想彻底重排历史才删该 manifest 文件。